# M11 — A generative model must reproduce more than a mean

<!-- paper-first -->
### Research question

**Reading:** [PM01](../../curriculum/papers/modeling.md#pm01). Review the assigned figure or result before starting the lesson.

**Question:** What observations should a proposed model generate if it is to explain more than a fitted mean?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

A discriminative model predicts an outcome conditional on features. A generative model describes a distribution from which observations could arise, often through latent variables. This gives it a different set of checks: not just whether it predicts labels, but whether simulated observations reproduce meaningful properties of measured data. Generative does not mean that synthetic outputs are automatically realistic, private, or suitable replacements for participants.

Factor analysis describes observations as a linear combination of latent Gaussian factors plus feature-specific noise. In a simple notation, x equals a mean plus a loading matrix times a latent vector plus residual noise. The factors can explain correlated variation among features, while the noise variances describe variation not explained by the shared factors. Unlike ordinary PCA reconstruction, the model supplies a probabilistic account of residual variation under its assumptions.

The experiment generates six observed features from two latent factors. We fit factor analysis using only the training rows, then compare the observed covariance with covariance from fresh draws under the fitted model. A successful mean or variance check is not sufficient: a model that independently resamples every feature can preserve marginal distributions while destroying relationships among features. The failure demonstration deliberately creates that independent marginal model.

Latent coordinates have ambiguities. Rotations of factors can produce equivalent observed covariance under suitable transformations, so naming a latent coordinate as a unique biological process requires additional evidence or constraints. Factor count is also a modeling choice. More factors can fit training covariance more closely while creating uncertainty or poor transfer. Select complexity with a procedure separated from final evaluation and inspect sensitivity to plausible alternatives.

Other generative families include Gaussian mixtures, variational autoencoders, autoregressive models, and diffusion models. A mixture models several component distributions; a variational autoencoder learns an encoder and decoder together with a latent-distribution objective; diffusion learns a denoising process across noise levels. These approaches require distinct objectives and evaluation. The offline lab runs factor analysis only; the Neuromatch generative tutorial is an explicit extension, not an execution claim.

Ask AI to generate a posterior-predictive or model-simulation check targeted to the scientific question: covariance, temporal structure, lesion-size distribution, or subgroup behavior. Distinguish simulating under fitted point parameters from integrating parameter uncertainty. Our check uses fitted point parameters and therefore does not express all model uncertainty. A convincing image can still contain unrealistic dependencies, and synthetic training data can amplify a generator's omissions.

## Transformation contract

Observed feature rows → fitted latent loadings and noise variances → latent draws plus noise → simulated observations. The latent representation compresses measurements; simulation reconstructs a distribution under assumptions, not the original people.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from sklearn.decomposition import FactorAnalysis
rng=np.random.default_rng(411)
loading=np.array([[1,0],[.8,.2],[1.2,0],[0,1],[.1,.9],[0,1.3]])
X=rng.normal(size=(1800,2))@loading.T+rng.normal(0,.25,(1800,6))
fa=FactorAnalysis(n_components=2,random_state=1).fit(X[:1200])
sim=rng.normal(size=(6000,2))@fa.components_+fa.mean_
sim+=rng.normal(size=sim.shape)*np.sqrt(fa.noise_variance_)
observed_cov=np.cov(X[1200:],rowvar=False);sim_cov=np.cov(sim,rowvar=False)
error=np.linalg.norm(observed_cov-sim_cov)/np.linalg.norm(observed_cov)
print('Relative held-out covariance error:',error)
assert np.all(fa.noise_variance_>0) and error<.2


Relative held-out covariance error: 0.10401020633255886


In [2]:
independent=rng.normal(size=(6000,6))*X[:1200].std(axis=0)+X[:1200].mean(axis=0)
bad_cov=np.cov(independent,rowvar=False)
bad_error=np.linalg.norm(observed_cov-bad_cov)/np.linalg.norm(observed_cov)
print('Independent-marginals covariance error:',bad_error)
assert bad_error>error+.2
print('Fitted factor dimensions:',fa.transform(X[1200:]).shape)
assert fa.transform(X[1200:]).shape==(600,2)


Independent-marginals covariance error: 0.7929942611000556
Fitted factor dimensions: (600, 2)


## Deliberate failure and repair

Matching each feature’s mean and variance while sampling it independently destroys shared covariance. Repair the generative assumptions or narrow the intended use; adding more synthetic rows from the wrong distribution does not fix it. Avoid describing the latent factors as uniquely identified biological entities.

## Your investigation

Choose two diagnostics a model of regional brain volumes should reproduce, beyond marginal means. Fit one and three factors and compare held-out log likelihood and covariance checks, keeping model selection separate from any final test. Explain which uncertainties are absent when simulations use only one fitted loading matrix.

## Transfer to real neuroimaging

The NMA extension includes neural generative models and data dependencies beyond this core. For MRI synthesis, evaluate spatial, acquisition, population, and privacy assumptions with appropriate experts. No VAE, diffusion model, or real MRI generator is trained here.

**Primary teaching sources, pinned where hosted on GitHub:**

- [NMA DL: generative models and latent variables](https://github.com/NeuromatchAcademy/course-content-dl/blob/caba36c513fb8139ac3c9e7503f7a769dadde25e/tutorials/W2D3_GenerativeModelsAndDeepLearningDiscussion1/student/W2D3_Tutorial1.ipynb)
- [scikit-learn factor-analysis documentation](https://scikit-learn.org/stable/modules/decomposition.html#factor-analysis)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Does matching marginal distributions guarantee realistic joint data? **No; dependence can be wrong.**
2. Are generated rows new independently recruited participants? **No; they are draws under the fitted model and its biases.**

### Return to the research question

Revisit [PM01](../../curriculum/papers/modeling.md#pm01) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
